# 4 结果处理与性能调优
第 3 章已生成检测结果。本章显示结果图，再编译并测试多流候选 OM。

如果仓库不在 CANN Lab 默认目录，请在第一个 Code Cell 中填写 `USER_REPO_ROOT`，或提前设置 `GITCODE_REPO_ROOT`。

In [ ]:
from pathlib import Path
import os, re, subprocess, sys
USER_REPO_ROOT = ''  # 可选：例如 '/mnt/workspace/my-repo'
DEFAULT_REPO_ROOT = '/mnt/workspace/gitCode/cann/cann-learning-hub'
REPO_ROOT = Path(USER_REPO_ROOT or os.environ.get('GITCODE_REPO_ROOT', DEFAULT_REPO_ROOT))
TUTORIAL_DIR=REPO_ROOT/'reference_practice'/'yolov13_offline_inference'; DATA_DIR=TUTORIAL_DIR/'data'
os.environ['GITCODE_REPO_ROOT']=str(REPO_ROOT)
os.environ['LD_LIBRARY_PATH']=str(Path(sys.prefix)/'lib')+':'+os.environ.get('LD_LIBRARY_PATH','')
from IPython.display import Image, display
subprocess.run(['python3',str(TUTORIAL_DIR/'scripts'/'draw_boxes.py')],cwd=DATA_DIR,check=True)
display(Image(filename=str(DATA_DIR/'bus_out.jpg'),width=640))

## 2. 自动识别目标芯片
候选 OM 也必须针对当前设备编译。这里沿用第 2 章的识别逻辑：先用 `npu-smi info -t board` 读取 `Chip Name` 和 `NPU Name`，两者缺失时再用普通 `npu-smi info` 的设备名称回退。未设置 `NPU_ID` 时自动选择 `npu-smi info -l` 返回的第一个设备；多卡环境也可以手动设置 `NPU_ID` 和 `CHIP_ID`。

In [ ]:
cann_path=os.environ.get('ASCEND_HOME_PATH')
if not cann_path: raise RuntimeError('ASCEND_HOME_PATH 未设置，请先加载 CANN 的 set_env.sh。')
CANN_ROOT=Path(cann_path); npu_env=os.environ.copy()
npu_env['LD_LIBRARY_PATH']=str(CANN_ROOT/'aarch64-linux'/'lib64')+':'+npu_env.get('LD_LIBRARY_PATH','')
npu_id=os.environ.get('NPU_ID')
if not npu_id:
    listed=subprocess.run(['npu-smi','info','-l'],env=npu_env,capture_output=True,text=True,check=True).stdout
    ids=re.findall(r'NPU ID\s*:\s*(\d+)',listed)
    if not ids: raise RuntimeError('无法从 npu-smi info -l 识别 NPU_ID，请设置 NPU_ID。')
    npu_id=ids[0]
chip_id=os.environ.get('CHIP_ID','0')
board=subprocess.run(['npu-smi','info','-t','board','-i',npu_id,'-c',chip_id],env=npu_env,capture_output=True,text=True,check=True).stdout
chip_name=re.search(r'Chip Name\s*:\s*(\S+)',board); npu_name=re.search(r'NPU Name\s*:\s*(\S+)',board)
if chip_name and npu_name:
    SOC_VERSION=f'{chip_name.group(1)}_{npu_name.group(1)}'
else:
    info=subprocess.run(['npu-smi','info'],env=npu_env,capture_output=True,text=True,check=True).stdout
    name=re.search(r'\|\s*\d+\s+(\S+)',info)
    if not name: raise RuntimeError('无法从 npu-smi 输出识别 soc_version，请设置 SOC_VERSION 手动覆盖。')
    SOC_VERSION=name.group(1) if name.group(1).startswith('Ascend') else 'Ascend'+name.group(1)
print(f'NPU_ID={npu_id}, CHIP_ID={chip_id}, SOC_VERSION={SOC_VERSION}')

## 3. 检查 CANN 版本
多流增强依赖 CANN 9.2.0 或更高版本。版本从 Toolkit 的 `opp/version.info` 或安装信息中读取；如果版本过低，Cell 只打印不支持提示，不会继续调用带多流参数的 ATC。

In [ ]:
version_files=[CANN_ROOT/'opp'/'version.info',CANN_ROOT/'aarch64-linux'/'ascend_toolkit_install.info']
version_text=next((p.read_text(encoding='utf-8',errors='ignore') for p in version_files if p.exists()),'')
version_match=re.search(r'(?:Version|version)\s*[=:]\s*([0-9]+(?:\.[0-9]+){1,2})',version_text)
cann_version=tuple(int(part) for part in version_match.group(1).split('.')) if version_match else (0,0,0)
MULTI_STREAM_SUPPORTED=cann_version >= (9,2,0)
version_name=version_match.group(1) if version_match else 'unknown'
if not MULTI_STREAM_SUPPORTED: print(f'CANN {version_name} 不支持多流候选 OM 编译，需要 CANN 9.2.0 或更高版本。')
else: print(f'CANN {version_name} 支持多流候选 OM 编译。')

## 4. 编译多流候选 OM
只有 CANN 版本满足要求时才执行下面的 ATC Cell；`MODE` 是唯一变化的参数。`--multi_stream_parallel_mode` 的其他取值和限制请参考 [ATC 参数说明](https://www.hiascend.com/document/detail/zh/canncommercial/latest/devaids/atctool/atlasatcparam_16_0036.html)。

In [ ]:
if MULTI_STREAM_SUPPORTED:
    MODE='cv'; ONNX_PATH=TUTORIAL_DIR/'workspace'/'yolov13n.onnx'; MODEL_DIR=TUTORIAL_DIR/'model'
    name=re.sub(r'[^A-Za-z0-9]+','_',MODE).strip('_').lower(); prefix=MODEL_DIR/('yolov13_'+name)
    subprocess.run(['atc','--model='+str(ONNX_PATH),'--framework=5','--output='+str(prefix),'--soc_version='+SOC_VERSION,'--input_shape=images:1,3,640,640','--output_type=FP32','--multi_stream_parallel_mode='+MODE],check=True)
    CANDIDATE_OM=prefix.with_suffix('.om')

## 5. 使用相同口径测量候选配置
CANN 版本满足要求并生成候选 OM 后，使用 10 次预热和 90 次正式推理；旧版本没有候选 OM 时跳过测量。

In [ ]:
if MULTI_STREAM_SUPPORTED:
    run=subprocess.run([str(TUTORIAL_DIR/'out'/'yolov13_main'),'--model='+str(CANDIDATE_OM),'--input='+str(DATA_DIR/'bus.bin'),'--warmup-runs=10','--runs=90'],cwd=TUTORIAL_DIR/'out',text=True,capture_output=True,check=True)
    print(run.stdout)
    latency=re.findall(r'Average aclmdlExecute latency: ([0-9.]+) ms',run.stdout)
    print('Average latency (ms):',latency[-1] if latency else 'not found')